# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 5.0 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64, pickle
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
torch.set_num_threads(1)

In [6]:
TASK_ID='task005'; H=W=30; CH=10
TASK_PATH=Path(COMPETITION)/'task005.json'
OUT=Path.cwd()/'task005_allsource_fast_attempt'; 
OUT.mkdir(exist_ok=True)
ONNX_PATH=OUT/'task005.onnx'
ZIP_PATH=Path.cwd()/'submission.zip'
PKG_PATH=OUT/'task005_allsource_fast_onnx_package.zip'
HEALTH_PATH=OUT/'task005_allsource_fast_health.json'
NB_PATH=OUT/'task005_allsource_fast_onnx.ipynb'
FORBIDDEN={'Loop','Scan','NonZero','Unique','Script','Function'}
DIRS=[(-4,-4),(-4,0),(-4,4),(0,-4),(0,4),(4,-4),(4,0),(4,4)]

In [7]:
def grid_to_tensor(grid):
    a=np.array(grid,dtype=np.int64); x=np.zeros((1,CH,H,W),np.float32)
    for r in range(a.shape[0]):
        for c in range(a.shape[1]): x[0,int(a[r,c]),r,c]=1.0
    return x

def tensor_to_grid(y,shape):
    return np.asarray(y)[0].argmax(0).astype(np.int64)[:shape[0],:shape[1]]

class Task005AllSourceFast(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('rows3',torch.arange(H,dtype=torch.float32).view(1,1,H))
        self.register_buffer('cols3',torch.arange(W,dtype=torch.float32).view(1,1,W))
        eye=torch.eye(9).view(1,9,9,1)
        self.register_buffer('not_eye',1.0-eye)
        # Tiny bias: prefer lower color if evidence score ties, after source pixel count.
        self.register_buffer('bias',torch.linspace(0.0009,0.0001,9).view(1,9,1,1))
    def sh(self,t,dr:int,dc:int):
        if abs(dr)>=H or abs(dc)>=W: return t*0.0
        if dr>0:
            z=t[:,:,:dr,:]*0.0; t=torch.cat([z,t[:,:,:H-dr,:]],2)
        elif dr<0:
            z=t[:,:,:-dr,:]*0.0; t=torch.cat([t[:,:,-dr:,:],z],2)
        if dc>0:
            z=t[:,:,:,:dc]*0.0; t=torch.cat([z,t[:,:,:,:W-dc]],3)
        elif dc<0:
            z=t[:,:,:,:-dc]*0.0; t=torch.cat([t[:,:,:,-dc:],z],3)
        return t
    def ray(self,src,dr:int,dc:int):
        acc=src*0.0
        for k in range(1,8):
            acc=acc+self.sh(src,dr*k,dc*k)
        return torch.clamp(acc,0,1)
    def compact3(self,p):
        total=p.sum((2,3),keepdim=True)
        row_has=(p.sum(3)>0.5).float(); col_has=(p.sum(2)>0.5).float()
        rmin=torch.where(row_has>0.5,self.rows3,self.rows3+99).amin(2).view(1,9,1,1)
        rmax=torch.where(row_has>0.5,self.rows3,self.rows3-99).amax(2).view(1,9,1,1)
        cmin=torch.where(col_has>0.5,self.cols3,self.cols3+99).amin(2).view(1,9,1,1)
        cmax=torch.where(col_has>0.5,self.cols3,self.cols3-99).amax(2).view(1,9,1,1)
        bh=rmax-rmin+1; bw=cmax-cmin+1
        return ((torch.abs(bh-3)<0.1)&(torch.abs(bw-3)<0.1)&(total>=4)&(total<=9.1)).float(), total
    def forward(self,x):
        p=x[:,1:]
        compact,total=self.compact3(p)
        rays=[]
        for si in range(9):
            src=p[:,si:si+1]
            for dr,dc in DIRS:
                rays.append(self.ray(src,dr,dc))
        rays=torch.cat(rays,1).view(1,9,8,H,W)
        # overlap[source,target,dir]
        ov=(rays.unsqueeze(2) * p.unsqueeze(1).unsqueeze(3)).sum(dim=(-1,-2))
        hit=(ov>0.5).float()*self.not_eye
        # source score: number of target-direction hits, with compact gate
        evid=hit.sum(dim=(2,3),keepdim=True) # (1,9,1,1)
        score=compact*evid + compact*total*0.001 + compact*self.bias
        mx=score.amax(1,keepdim=True)
        sel=((score>=mx-1e-5)&(mx>0.5)).float()
        # selected source candidate output per target color
        weights=hit*sel # (1,9,9,8)
        add=(weights[...,None,None]*rays.unsqueeze(2)).sum(dim=(1,3)) # (1,9,H,W)
        add=torch.clamp(add,0,1)*x[:,0:1]
        anyadd=torch.clamp(add.sum(1,keepdim=True),0,1)
        y0=x[:,0:1]*(1-anyadd)
        yrest=torch.clamp(p+add,0,1)
        return torch.cat([y0,yrest],1)

def eval_examples(sess,examples):
    name=sess.get_inputs()[0].name; exact=0; total=0; first=None
    for i,ex in enumerate(examples):
        if 'output' not in ex: continue
        total+=1; out=np.array(ex['output'],dtype=np.int64)
        pred=tensor_to_grid(sess.run(None,{name:grid_to_tensor(ex['input'])})[0],out.shape)
        ok=np.array_equal(pred,out); exact+=int(ok)
        if not ok and first is None:
            first={'idx':i,'wrong_pixels':int((pred!=out).sum())}
    return {'exact':exact,'total':total,'first_wrong':first}


In [8]:
data=json.load(open(TASK_PATH))
model=Task005AllSourceFast().eval()
dummy=torch.zeros(1,CH,H,W); dummy[:,0]=1
for pth in [ONNX_PATH,ZIP_PATH,PKG_PATH,HEALTH_PATH,NB_PATH]:
    try: Path(pth).unlink()
    except FileNotFoundError: pass
st=time.time(); torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],opset_version=17,dynamic_axes=None,do_constant_folding=True,dynamo=False); export_time=time.time()-st


/tmp/ipykernel_16/311160249.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  st=time.time(); torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],opset_version=17,dynamic_axes=None,do_constant_folding=True,dynamo=False); export_time=time.time()-st


In [9]:
m=onnx.load(str(ONNX_PATH)); onnx.checker.check_model(m); ops=collections.Counter(n.op_type for n in m.graph.node)
sess=ort.InferenceSession(str(ONNX_PATH),providers=['CPUExecutionProvider'])
arc=data['arc-gen']; split=int(.4*len(arc))
health={'task':TASK_ID,'model':'all-source fast source selector plus 8-dir ray completion','onnx_size_bytes':ONNX_PATH.stat().st_size,'export_time_seconds':export_time,'ops':dict(ops),'forbidden_ops':sorted(set(ops)&FORBIDDEN),'input_shape':[(d.dim_value or d.dim_param) for d in m.graph.input[0].type.tensor_type.shape.dim],'output_shape':[(d.dim_value or d.dim_param) for d in m.graph.output[0].type.tensor_type.shape.dim],'train':eval_examples(sess,data['train']),'visible_test':eval_examples(sess,data['test']),'arc_gen_fit_40':eval_examples(sess,arc[:split]),'arc_gen_holdout_60':eval_examples(sess,arc[split:]),'arc_gen_full':eval_examples(sess,arc),'exact_lookup_bank':False,'tree_based_method':False,'zip_entries':['task005.onnx']}
print(json.dumps(health,indent=2)[:5000])
json.dump(health,open(HEALTH_PATH,'w'),indent=2)

{
  "task": "task005",
  "model": "all-source fast source selector plus 8-dir ray completion",
  "onnx_size_bytes": 529050,
  "export_time_seconds": 14.795108556747437,
  "ops": {
    "Identity": 3,
    "Constant": 3512,
    "Slice": 767,
    "ReduceSum": 7,
    "Greater": 6,
    "Cast": 5,
    "Where": 4,
    "ReduceMin": 2,
    "Reshape": 5,
    "ReduceMax": 3,
    "Sub": 6,
    "Add": 509,
    "Abs": 2,
    "Less": 2,
    "And": 4,
    "GreaterOrEqual": 2,
    "LessOrEqual": 1,
    "Mul": 271,
    "Concat": 506,
    "Clip": 75,
    "Unsqueeze": 5
  },
  "forbidden_ops": [],
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "train": {
    "exact": 3,
    "total": 3,
    "first_wrong": null
  },
  "visible_test": {
    "exact": 1,
    "total": 1,
    "first_wrong": null
  },
  "arc_gen_fit_40": {
    "exact": 104,
    "total": 104,
    "first_wrong": null
  },
  "arc_gen_holdout_60": {
    "exact": 158,
    "total": 158,
   

In [10]:
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as z: 
    z.write(ONNX_PATH,'task005.onnx')